# Práctica SQL 02 — Caso 01: E-commerce Andino (preguntas)

10 preguntas básicas y 20 intermedias.

**Plataforma:** Databricks SQL · Delta Lake · Unity Catalog

## Descripción

Una empresa peruana de comercio electrónico vende productos de varias categorías mediante web, app y tiendas físicas. La gerencia necesita analizar clientes, pedidos, vendedores, ventas, descuentos, tiempos de entrega y devoluciones.

### Regla de negocio para los ejercicios

Salvo que la pregunta indique lo contrario, considera como **venta válida** todo pedido cuyo estado sea diferente de `cancelado`. El importe neto de una línea se calcula así:

```sql
cantidad * precio_unitario * (1 - descuento_pct / 100)
```

## Tablas

| Tabla | Columnas y tipos | Descripción |
|---|---|---|
| `clientes` | `cliente_id STRING`; `nombre_cliente STRING`; `ciudad STRING`; `region STRING`; `segmento STRING`; `fecha_registro DATE`; `email STRING` | Información demográfica y fecha de registro. |
| `vendedores` | `vendedor_id STRING`; `nombre_vendedor STRING`; `region STRING`; `fecha_ingreso DATE`; `meta_mensual DECIMAL(12,2)` | Región y meta mensual de cada vendedor. |
| `productos` | `producto_id STRING`; `nombre_producto STRING`; `categoria STRING`; `marca STRING`; `precio_lista DECIMAL(12,2)`; `costo_unitario DECIMAL(12,2)`; `activo BOOLEAN` | Catálogo, precios y costos. |
| `pedidos` | `pedido_id STRING`; `cliente_id STRING`; `vendedor_id STRING`; `fecha_pedido DATE`; `canal STRING`; `metodo_pago STRING`; `estado STRING`; `ciudad_envio STRING`; `fecha_entrega DATE` *(admite NULL)* | Cabecera del pedido, canal, estado y entrega. |
| `detalle_pedidos` | `pedido_id STRING`; `linea_id INT`; `producto_id STRING`; `cantidad INT`; `precio_unitario DECIMAL(12,2)`; `descuento_pct DECIMAL(5,2)` | Productos, cantidades, precios y descuentos. |
| `devoluciones` | `devolucion_id STRING`; `pedido_id STRING`; `producto_id STRING`; `fecha_devolucion DATE`; `cantidad_devuelta INT`; `motivo STRING`; `monto_reembolso DECIMAL(12,2)` | Productos devueltos y montos reembolsados. |

## Relaciones

- `clientes.cliente_id` → `pedidos.cliente_id` (1 a N)
- `vendedores.vendedor_id` → `pedidos.vendedor_id` (1 a N)
- `pedidos.pedido_id` → `detalle_pedidos.pedido_id` (1 a N)
- `productos.producto_id` → `detalle_pedidos.producto_id` (1 a N)
- `pedidos.pedido_id` → `devoluciones.pedido_id` (1 a N)
- `productos.producto_id` → `devoluciones.producto_id` (1 a N)

## Consideraciones

- Los archivos TXT usan formato CSV con encabezado y codificación UTF-8.
- Las fechas usan el formato `YYYY-MM-DD`.
- Los campos vacíos representan `NULL`, principalmente `fecha_entrega` (pedidos aún no entregados).
- Los pedidos cancelados conservan sus líneas para permitir analizar intentos de compra, pero **no** se consideran ventas válidas.

---
## PARTE 0 — Preparación del entorno (DDL y carga)

> Si ya creaste y cargaste las tablas de este caso, salta directamente a la **PARTE 1**.

### 0.1 Esquemas y volumen

| Objeto | Nombre | Para qué sirve |
|---|---|---|
| Esquema | `workspace.sql_practice_raw` | Contiene el volumen con los archivos originales. |
| Volumen | `workspace.sql_practice_raw.datasets` | Almacena los `.txt` de ambos casos, cada uno en su carpeta. |
| Esquema | `workspace.sql_practice_ecommerce` | Contiene las 6 tablas de este caso. |

En Databricks Free Edition el catálogo por defecto es `workspace`. Si el tuyo se llama `main` u otro nombre, reemplaza `workspace` en todo el notebook, incluidas las rutas `/Volumes/workspace/...`.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.sql_practice_raw;
CREATE VOLUME IF NOT EXISTS workspace.sql_practice_raw.datasets;

CREATE SCHEMA IF NOT EXISTS workspace.sql_practice_ecommerce;

### 0.2 Subir los archivos al volumen

1. En **Catalog Explorer** abre el volumen `workspace` → `sql_practice_raw` → **Volumes** → `datasets`.
2. Crea la carpeta **`source01`** (si aún no existe).
3. Dentro de esa carpeta, sube los 6 archivos de tu carpeta local `source01/`: `clientes.txt`, `vendedores.txt`, `productos.txt`, `pedidos.txt`, `detalle_pedidos.txt`, `devoluciones.txt`.
4. Ejecuta la siguiente celda: deben aparecer los 6 archivos.

In [ ]:
%sql
LIST '/Volumes/workspace/sql_practice_raw/datasets/source01/';

### 0.3 Eliminar tablas previas

Permite volver a ejecutar la preparación desde cero. Las tablas se eliminan en **orden inverso a sus dependencias**: primero las hijas y luego las padres.

In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA sql_practice_ecommerce;

DROP TABLE IF EXISTS devoluciones;
DROP TABLE IF EXISTS detalle_pedidos;
DROP TABLE IF EXISTS pedidos;
DROP TABLE IF EXISTS productos;
DROP TABLE IF EXISTS vendedores;
DROP TABLE IF EXISTS clientes;

### 0.4 Crear las tablas (DDL)

- Todas las columnas son `NOT NULL`, salvo las indicadas como *admite NULL*. Delta Lake **sí** valida esta restricción al insertar.
- `PRIMARY KEY` y `FOREIGN KEY` en Unity Catalog son restricciones **informativas**: documentan el modelo, pero **no se validan** al insertar. Por eso en el paso 0.6 verificamos la integridad con consultas.
- Las tablas se crean de **padres a hijas**: una clave foránea solo puede apuntar a una tabla que ya tenga su clave primaria.

#### `clientes`

In [ ]:
%sql
CREATE TABLE clientes (
  cliente_id      STRING  NOT NULL COMMENT 'Identificador del cliente',
  nombre_cliente  STRING  NOT NULL COMMENT 'Nombre completo',
  ciudad          STRING  NOT NULL COMMENT 'Ciudad de residencia',
  region          STRING  NOT NULL COMMENT 'Región del país',
  segmento        STRING  NOT NULL COMMENT 'Segmento del cliente',
  fecha_registro  DATE    NOT NULL COMMENT 'Fecha de registro',
  email           STRING  NOT NULL COMMENT 'Correo electrónico (ficticio)',
  CONSTRAINT pk_clientes PRIMARY KEY (cliente_id)
)
COMMENT 'Clientes de la tienda online.';

#### `vendedores`

In [ ]:
%sql
CREATE TABLE vendedores (
  vendedor_id      STRING         NOT NULL COMMENT 'Identificador del vendedor',
  nombre_vendedor  STRING         NOT NULL COMMENT 'Nombre completo',
  region           STRING         NOT NULL COMMENT 'Región asignada',
  fecha_ingreso    DATE           NOT NULL COMMENT 'Fecha de ingreso a la empresa',
  meta_mensual     DECIMAL(12,2)  NOT NULL COMMENT 'Meta de ventas mensual en soles',
  CONSTRAINT pk_vendedores PRIMARY KEY (vendedor_id)
)
COMMENT 'Vendedores y su meta mensual.';

#### `productos`

In [ ]:
%sql
CREATE TABLE productos (
  producto_id      STRING         NOT NULL COMMENT 'Identificador del producto',
  nombre_producto  STRING         NOT NULL COMMENT 'Nombre del producto',
  categoria        STRING         NOT NULL COMMENT 'Categoría del producto',
  marca            STRING         NOT NULL COMMENT 'Marca',
  precio_lista     DECIMAL(12,2)  NOT NULL COMMENT 'Precio de lista en soles',
  costo_unitario   DECIMAL(12,2)  NOT NULL COMMENT 'Costo unitario en soles',
  activo           BOOLEAN        NOT NULL COMMENT 'Indica si el producto está activo',
  CONSTRAINT pk_productos PRIMARY KEY (producto_id)
)
COMMENT 'Catálogo de productos.';

#### `pedidos`

In [ ]:
%sql
CREATE TABLE pedidos (
  pedido_id      STRING  NOT NULL COMMENT 'Identificador del pedido',
  cliente_id     STRING  NOT NULL COMMENT 'Cliente que realiza el pedido',
  vendedor_id    STRING  NOT NULL COMMENT 'Vendedor responsable',
  fecha_pedido   DATE    NOT NULL COMMENT 'Fecha del pedido',
  canal          STRING  NOT NULL COMMENT 'Canal de venta',
  metodo_pago    STRING  NOT NULL COMMENT 'Método de pago',
  estado         STRING  NOT NULL COMMENT 'Estado del pedido',
  ciudad_envio   STRING  NOT NULL COMMENT 'Ciudad de envío',
  fecha_entrega  DATE             COMMENT 'Fecha de entrega (NULL si aún no se entrega)',
  CONSTRAINT pk_pedidos PRIMARY KEY (pedido_id),
  CONSTRAINT fk_pedidos_clientes FOREIGN KEY (cliente_id) REFERENCES clientes (cliente_id),
  CONSTRAINT fk_pedidos_vendedores FOREIGN KEY (vendedor_id) REFERENCES vendedores (vendedor_id)
)
COMMENT 'Cabecera de pedidos.';

#### `detalle_pedidos`

In [ ]:
%sql
CREATE TABLE detalle_pedidos (
  pedido_id        STRING         NOT NULL COMMENT 'Pedido al que pertenece la línea',
  linea_id         INT            NOT NULL COMMENT 'Número de línea dentro del pedido',
  producto_id      STRING         NOT NULL COMMENT 'Producto vendido',
  cantidad         INT            NOT NULL COMMENT 'Unidades vendidas',
  precio_unitario  DECIMAL(12,2)  NOT NULL COMMENT 'Precio unitario aplicado en soles',
  descuento_pct    DECIMAL(5,2)   NOT NULL COMMENT 'Descuento en porcentaje (0 a 100)',
  CONSTRAINT pk_detalle_pedidos PRIMARY KEY (pedido_id, linea_id),
  CONSTRAINT fk_detalle_pedidos_pedidos FOREIGN KEY (pedido_id) REFERENCES pedidos (pedido_id),
  CONSTRAINT fk_detalle_pedidos_productos FOREIGN KEY (producto_id) REFERENCES productos (producto_id)
)
COMMENT 'Líneas de cada pedido.';

#### `devoluciones`

In [ ]:
%sql
CREATE TABLE devoluciones (
  devolucion_id      STRING         NOT NULL COMMENT 'Identificador de la devolución',
  pedido_id          STRING         NOT NULL COMMENT 'Pedido de origen',
  producto_id        STRING         NOT NULL COMMENT 'Producto devuelto',
  fecha_devolucion   DATE           NOT NULL COMMENT 'Fecha de la devolución',
  cantidad_devuelta  INT            NOT NULL COMMENT 'Unidades devueltas',
  motivo             STRING         NOT NULL COMMENT 'Motivo de la devolución',
  monto_reembolso    DECIMAL(12,2)  NOT NULL COMMENT 'Monto reembolsado en soles',
  CONSTRAINT pk_devoluciones PRIMARY KEY (devolucion_id),
  CONSTRAINT fk_devoluciones_pedidos FOREIGN KEY (pedido_id) REFERENCES pedidos (pedido_id),
  CONSTRAINT fk_devoluciones_productos FOREIGN KEY (producto_id) REFERENCES productos (producto_id)
)
COMMENT 'Productos devueltos y montos reembolsados.';

### 0.5 Cargar los datos

Usamos `read_files` para leer cada `.txt` del volumen:

- `format => 'csv'` y `header => true`: texto delimitado por comas con encabezado.
- `schema => '...'`: define el tipo de cada columna en lugar de dejar que Databricks lo infiera. Los campos vacíos se cargan como `NULL`.
- `mode => 'FAILFAST'`: si alguna fila no se puede convertir al tipo indicado, la carga falla en lugar de dejar datos incorrectos.

#### `clientes`

In [ ]:
%sql
INSERT INTO clientes
SELECT
  cliente_id,
  nombre_cliente,
  ciudad,
  region,
  segmento,
  fecha_registro,
  email
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/source01/clientes.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'cliente_id STRING, nombre_cliente STRING, ciudad STRING, region STRING, segmento STRING, fecha_registro DATE, email STRING'
);

SELECT * FROM clientes LIMIT 5;

#### `vendedores`

In [ ]:
%sql
INSERT INTO vendedores
SELECT
  vendedor_id,
  nombre_vendedor,
  region,
  fecha_ingreso,
  meta_mensual
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/source01/vendedores.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'vendedor_id STRING, nombre_vendedor STRING, region STRING, fecha_ingreso DATE, meta_mensual DECIMAL(12,2)'
);

SELECT * FROM vendedores LIMIT 5;

#### `productos`

In [ ]:
%sql
INSERT INTO productos
SELECT
  producto_id,
  nombre_producto,
  categoria,
  marca,
  precio_lista,
  costo_unitario,
  activo
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/source01/productos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'producto_id STRING, nombre_producto STRING, categoria STRING, marca STRING, precio_lista DECIMAL(12,2), costo_unitario DECIMAL(12,2), activo BOOLEAN'
);

SELECT * FROM productos LIMIT 5;

#### `pedidos`

In [ ]:
%sql
INSERT INTO pedidos
SELECT
  pedido_id,
  cliente_id,
  vendedor_id,
  fecha_pedido,
  canal,
  metodo_pago,
  estado,
  ciudad_envio,
  fecha_entrega
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/source01/pedidos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'pedido_id STRING, cliente_id STRING, vendedor_id STRING, fecha_pedido DATE, canal STRING, metodo_pago STRING, estado STRING, ciudad_envio STRING, fecha_entrega DATE'
);

SELECT * FROM pedidos LIMIT 5;

#### `detalle_pedidos`

In [ ]:
%sql
INSERT INTO detalle_pedidos
SELECT
  pedido_id,
  linea_id,
  producto_id,
  cantidad,
  precio_unitario,
  descuento_pct
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/source01/detalle_pedidos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'pedido_id STRING, linea_id INT, producto_id STRING, cantidad INT, precio_unitario DECIMAL(12,2), descuento_pct DECIMAL(5,2)'
);

SELECT * FROM detalle_pedidos LIMIT 5;

#### `devoluciones`

In [ ]:
%sql
INSERT INTO devoluciones
SELECT
  devolucion_id,
  pedido_id,
  producto_id,
  fecha_devolucion,
  cantidad_devuelta,
  motivo,
  monto_reembolso
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/source01/devoluciones.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'devolucion_id STRING, pedido_id STRING, producto_id STRING, fecha_devolucion DATE, cantidad_devuelta INT, motivo STRING, monto_reembolso DECIMAL(12,2)'
);

SELECT * FROM devoluciones LIMIT 5;

### 0.6 Validación

Todas las tablas deben mostrar `estado = OK`.

In [ ]:
%sql
WITH conteos AS (
  SELECT 'clientes' AS tabla, COUNT(*) AS filas, 120 AS filas_esperadas FROM clientes
  UNION ALL
  SELECT 'vendedores' AS tabla, COUNT(*) AS filas, 12 AS filas_esperadas FROM vendedores
  UNION ALL
  SELECT 'productos' AS tabla, COUNT(*) AS filas, 45 AS filas_esperadas FROM productos
  UNION ALL
  SELECT 'pedidos' AS tabla, COUNT(*) AS filas, 360 AS filas_esperadas FROM pedidos
  UNION ALL
  SELECT 'detalle_pedidos' AS tabla, COUNT(*) AS filas, 1030 AS filas_esperadas FROM detalle_pedidos
  UNION ALL
  SELECT 'devoluciones' AS tabla, COUNT(*) AS filas, 63 AS filas_esperadas FROM devoluciones
)
SELECT
  tabla,
  filas,
  filas_esperadas,
  IF(filas = filas_esperadas, 'OK', 'REVISAR') AS estado
FROM conteos;

Integridad referencial: como las claves foráneas son informativas, comprobamos que no existan filas hijas que apunten a un padre inexistente. Todas las relaciones deben mostrar `huerfanos = 0`.

In [ ]:
%sql
SELECT 'pedidos.cliente_id -> clientes' AS relacion, COUNT(*) AS huerfanos
FROM pedidos AS h
LEFT JOIN clientes AS p ON h.cliente_id = p.cliente_id
WHERE p.cliente_id IS NULL
UNION ALL
SELECT 'pedidos.vendedor_id -> vendedores' AS relacion, COUNT(*) AS huerfanos
FROM pedidos AS h
LEFT JOIN vendedores AS p ON h.vendedor_id = p.vendedor_id
WHERE p.vendedor_id IS NULL
UNION ALL
SELECT 'detalle_pedidos.pedido_id -> pedidos' AS relacion, COUNT(*) AS huerfanos
FROM detalle_pedidos AS h
LEFT JOIN pedidos AS p ON h.pedido_id = p.pedido_id
WHERE p.pedido_id IS NULL
UNION ALL
SELECT 'detalle_pedidos.producto_id -> productos' AS relacion, COUNT(*) AS huerfanos
FROM detalle_pedidos AS h
LEFT JOIN productos AS p ON h.producto_id = p.producto_id
WHERE p.producto_id IS NULL
UNION ALL
SELECT 'devoluciones.pedido_id -> pedidos' AS relacion, COUNT(*) AS huerfanos
FROM devoluciones AS h
LEFT JOIN pedidos AS p ON h.pedido_id = p.pedido_id
WHERE p.pedido_id IS NULL
UNION ALL
SELECT 'devoluciones.producto_id -> productos' AS relacion, COUNT(*) AS huerfanos
FROM devoluciones AS h
LEFT JOIN productos AS p ON h.producto_id = p.producto_id
WHERE p.producto_id IS NULL;

---
## Configuración para resolver las preguntas

Ejecuta esta celda cada vez que abras el notebook: las consultas usan nombres de tablas sin catálogo ni esquema.

In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA sql_practice_ecommerce;

SHOW TABLES;

---
## PARTE 1 — Preguntas básicas (1-10)

### 1. Todos los clientes

Muestra todos los registros de la tabla `clientes`.

In [ ]:
%sql
SELECT *
FROM clientes;

### 2. Productos activos

Lista el ID, nombre, categoría y precio de los productos activos.

In [ ]:
%sql
SELECT
  producto_id,
  nombre_producto,
  categoria,
  precio_lista
FROM productos
WHERE activo = true
ORDER BY producto_id;

### 3. Pedidos entregados

Muestra los pedidos cuyo estado sea `entregado`.

In [ ]:
%sql
SELECT *
FROM pedidos
WHERE estado = 'entregado'
ORDER BY fecha_pedido, pedido_id;

### 4. Ciudades únicas de clientes

Obtén la lista de ciudades únicas de los clientes, ordenadas alfabéticamente.

In [ ]:
%sql
SELECT DISTINCT ciudad
FROM clientes
ORDER BY ciudad;

### 5. Diez productos con mayor precio

Muestra los 10 productos con mayor precio de lista.

In [ ]:
%sql
SELECT
  producto_id,
  nombre_producto,
  categoria,
  precio_lista
FROM productos
ORDER BY precio_lista DESC
LIMIT 10;

### 6. Cantidad total de pedidos

Cuenta cuántos pedidos existen en total.

In [ ]:
%sql
SELECT COUNT(*) AS total_pedidos
FROM pedidos;

### 7. Precio promedio de los productos

Calcula el precio de lista promedio de todos los productos.

In [ ]:
%sql
SELECT ROUND(AVG(precio_lista), 2) AS precio_promedio
FROM productos;

### 8. Pedidos de enero de 2026

Muestra los pedidos realizados durante enero de 2026.

In [ ]:
%sql
-- Rango semiabierto [1 ene, 1 feb): no depende de cuántos días tiene el mes
SELECT *
FROM pedidos
WHERE fecha_pedido >= DATE '2026-01-01'
  AND fecha_pedido <  DATE '2026-02-01'
ORDER BY fecha_pedido, pedido_id;

### 9. Clientes por segmento

Cuenta cuántos clientes existen por segmento.

In [ ]:
%sql
SELECT
  segmento,
  COUNT(*) AS cantidad_clientes
FROM clientes
GROUP BY segmento
ORDER BY cantidad_clientes DESC;

### 10. Importe neto por línea

Muestra cada línea de pedido incluyendo una columna calculada `importe_neto_linea`.

In [ ]:
%sql
SELECT
  pedido_id,
  linea_id,
  producto_id,
  cantidad,
  precio_unitario,
  descuento_pct,
  ROUND(cantidad * precio_unitario * (1 - descuento_pct / 100), 2) AS importe_neto_linea
FROM detalle_pedidos
ORDER BY pedido_id, linea_id;

---
## PARTE 2 — Preguntas intermedias (11-30)

### 11. Clientes con cantidad de pedidos, incluyendo clientes sin pedidos

Muestra cada cliente con su cantidad total de pedidos, incluyendo clientes sin pedidos.

In [ ]:
%sql
-- LEFT JOIN conserva a los clientes sin pedidos; COUNT(p.pedido_id) no cuenta los NULL, así que les asigna 0
SELECT
  c.cliente_id,
  c.nombre_cliente,
  COUNT(p.pedido_id) AS cantidad_pedidos
FROM clientes AS c
LEFT JOIN pedidos AS p
  ON c.cliente_id = p.cliente_id
GROUP BY c.cliente_id, c.nombre_cliente
ORDER BY cantidad_pedidos DESC, c.cliente_id;

### 12. Ventas netas válidas por categoría

Calcula las ventas netas válidas por categoría de producto.

In [ ]:
%sql
SELECT
  pr.categoria,
  ROUND(SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)), 2) AS ventas_netas
FROM pedidos AS p
JOIN detalle_pedidos AS d
  ON p.pedido_id = d.pedido_id
JOIN productos AS pr
  ON d.producto_id = pr.producto_id
WHERE p.estado <> 'cancelado'
GROUP BY pr.categoria
ORDER BY ventas_netas DESC;

### 13. Ticket promedio de pedidos entregados

Calcula el ticket promedio de los pedidos entregados. Primero suma cada pedido y luego promedia esos totales.

In [ ]:
%sql
-- Paso 1: total de cada pedido entregado. Paso 2: promedio de esos totales.
WITH totales_pedido AS (
  SELECT
    p.pedido_id,
    SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)) AS total_pedido
  FROM pedidos AS p
  JOIN detalle_pedidos AS d
    ON p.pedido_id = d.pedido_id
  WHERE p.estado = 'entregado'
  GROUP BY p.pedido_id
)
SELECT
  COUNT(*) AS pedidos_entregados,
  ROUND(AVG(total_pedido), 2) AS ticket_promedio
FROM totales_pedido;

### 14. Pedidos válidos y ventas por mes

Obtén la cantidad de pedidos válidos y las ventas netas por mes.

In [ ]:
%sql
-- COUNT(DISTINCT): el JOIN repite el pedido una vez por cada línea de detalle
SELECT
  date_format(p.fecha_pedido, 'yyyy-MM') AS mes,
  COUNT(DISTINCT p.pedido_id) AS pedidos_validos,
  ROUND(SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)), 2) AS ventas_netas
FROM pedidos AS p
JOIN detalle_pedidos AS d
  ON p.pedido_id = d.pedido_id
WHERE p.estado <> 'cancelado'
GROUP BY date_format(p.fecha_pedido, 'yyyy-MM')
ORDER BY mes;

### 15. Top 10 clientes por gasto neto

Identifica los 10 clientes con mayor gasto neto en pedidos válidos.

In [ ]:
%sql
SELECT
  c.cliente_id,
  c.nombre_cliente,
  ROUND(SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)), 2) AS gasto_neto
FROM pedidos AS p
JOIN detalle_pedidos AS d
  ON p.pedido_id = d.pedido_id
JOIN clientes AS c
  ON p.cliente_id = c.cliente_id
WHERE p.estado <> 'cancelado'
GROUP BY c.cliente_id, c.nombre_cliente
ORDER BY gasto_neto DESC
LIMIT 10;

### 16. Productos nunca vendidos

Lista los productos que nunca aparecen en `detalle_pedidos`.

In [ ]:
%sql
-- Anti-join: productos sin ninguna fila coincidente en detalle_pedidos
SELECT
  pr.producto_id,
  pr.nombre_producto,
  pr.categoria,
  pr.activo
FROM productos AS pr
LEFT JOIN detalle_pedidos AS d
  ON pr.producto_id = d.producto_id
WHERE d.producto_id IS NULL
ORDER BY pr.producto_id;

### 17. Clientes sin pedidos

Lista los clientes que nunca realizaron un pedido.

In [ ]:
%sql
SELECT
  c.cliente_id,
  c.nombre_cliente,
  c.ciudad,
  c.segmento,
  c.fecha_registro
FROM clientes AS c
WHERE NOT EXISTS (
  SELECT 1
  FROM pedidos AS p
  WHERE p.cliente_id = c.cliente_id
)
ORDER BY c.cliente_id;

### 18. Promedio de días de entrega por canal

Calcula el promedio de días de entrega por canal para pedidos entregados.

In [ ]:
%sql
-- datediff(fin, inicio) devuelve los días entre ambas fechas
SELECT
  canal,
  COUNT(*) AS pedidos_entregados,
  ROUND(AVG(datediff(fecha_entrega, fecha_pedido)), 2) AS dias_promedio_entrega
FROM pedidos
WHERE estado = 'entregado'
  AND fecha_entrega IS NOT NULL
GROUP BY canal
ORDER BY dias_promedio_entrega;

### 19. Unidades vendidas, devueltas y porcentaje de devolución por categoría

Calcula por categoría: unidades vendidas, unidades devueltas y porcentaje de devolución sobre unidades vendidas. Considera ventas válidas.

In [ ]:
%sql
-- Se agrega cada lado por separado antes de unirlos: si se une devoluciones
-- directamente con detalle_pedidos, las cantidades se multiplican.
WITH vendidas AS (
  SELECT
    pr.categoria,
    SUM(d.cantidad) AS unidades_vendidas
  FROM pedidos AS p
  JOIN detalle_pedidos AS d
    ON p.pedido_id = d.pedido_id
  JOIN productos AS pr
    ON d.producto_id = pr.producto_id
  WHERE p.estado <> 'cancelado'
  GROUP BY pr.categoria
),
devueltas AS (
  SELECT
    pr.categoria,
    SUM(dv.cantidad_devuelta) AS unidades_devueltas
  FROM devoluciones AS dv
  JOIN pedidos AS p
    ON dv.pedido_id = p.pedido_id
  JOIN productos AS pr
    ON dv.producto_id = pr.producto_id
  WHERE p.estado <> 'cancelado'
  GROUP BY pr.categoria
)
SELECT
  v.categoria,
  v.unidades_vendidas,
  COALESCE(dv.unidades_devueltas, 0) AS unidades_devueltas,
  ROUND(COALESCE(dv.unidades_devueltas, 0) * 100.0 / v.unidades_vendidas, 2) AS pct_devolucion
FROM vendidas AS v
LEFT JOIN devueltas AS dv
  ON v.categoria = dv.categoria
ORDER BY pct_devolucion DESC;

### 20. Ventas mensuales por vendedor y cumplimiento de meta

Calcula las ventas mensuales por vendedor y el porcentaje de cumplimiento de su meta mensual.

In [ ]:
%sql
WITH ventas_vendedor_mes AS (
  SELECT
    p.vendedor_id,
    date_format(p.fecha_pedido, 'yyyy-MM') AS mes,
    SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)) AS ventas_netas
  FROM pedidos AS p
  JOIN detalle_pedidos AS d
    ON p.pedido_id = d.pedido_id
  WHERE p.estado <> 'cancelado'
  GROUP BY p.vendedor_id, date_format(p.fecha_pedido, 'yyyy-MM')
)
SELECT
  v.vendedor_id,
  v.nombre_vendedor,
  vm.mes,
  ROUND(vm.ventas_netas, 2) AS ventas_netas,
  v.meta_mensual,
  ROUND(vm.ventas_netas * 100 / v.meta_mensual, 2) AS pct_cumplimiento
FROM ventas_vendedor_mes AS vm
JOIN vendedores AS v
  ON vm.vendedor_id = v.vendedor_id
ORDER BY v.vendedor_id, vm.mes;

### 21. Ranking de productos por ventas dentro de cada categoría

Genera un ranking de productos por ventas netas dentro de cada categoría.

In [ ]:
%sql
WITH ventas_producto AS (
  SELECT
    pr.categoria,
    pr.producto_id,
    pr.nombre_producto,
    SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)) AS ventas_netas
  FROM pedidos AS p
  JOIN detalle_pedidos AS d
    ON p.pedido_id = d.pedido_id
  JOIN productos AS pr
    ON d.producto_id = pr.producto_id
  WHERE p.estado <> 'cancelado'
  GROUP BY pr.categoria, pr.producto_id, pr.nombre_producto
)
SELECT
  categoria,
  producto_id,
  nombre_producto,
  ROUND(ventas_netas, 2) AS ventas_netas,
  RANK() OVER (PARTITION BY categoria ORDER BY ventas_netas DESC) AS ranking
FROM ventas_producto
ORDER BY categoria, ranking;

### 22. Ventas mensuales y acumulado

Calcula las ventas netas mensuales y el acumulado de ventas desde el primer mes.

In [ ]:
%sql
WITH ventas_mes AS (
  SELECT
    date_format(p.fecha_pedido, 'yyyy-MM') AS mes,
    SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)) AS ventas_netas
  FROM pedidos AS p
  JOIN detalle_pedidos AS d
    ON p.pedido_id = d.pedido_id
  WHERE p.estado <> 'cancelado'
  GROUP BY date_format(p.fecha_pedido, 'yyyy-MM')
)
SELECT
  mes,
  ROUND(ventas_netas, 2) AS ventas_netas,
  ROUND(SUM(ventas_netas) OVER (ORDER BY mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS ventas_acumuladas
FROM ventas_mes
ORDER BY mes;

### 23. Crecimiento porcentual respecto al mes anterior

Calcula el crecimiento porcentual de ventas netas respecto al mes anterior.

In [ ]:
%sql
WITH ventas_mes AS (
  SELECT
    date_format(p.fecha_pedido, 'yyyy-MM') AS mes,
    SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)) AS ventas_netas
  FROM pedidos AS p
  JOIN detalle_pedidos AS d
    ON p.pedido_id = d.pedido_id
  WHERE p.estado <> 'cancelado'
  GROUP BY date_format(p.fecha_pedido, 'yyyy-MM')
),
con_mes_anterior AS (
  SELECT
    mes,
    ventas_netas,
    LAG(ventas_netas) OVER (ORDER BY mes) AS ventas_mes_anterior
  FROM ventas_mes
)
-- El primer mes no tiene mes anterior: su crecimiento queda en NULL
SELECT
  mes,
  ROUND(ventas_netas, 2) AS ventas_netas,
  ROUND(ventas_mes_anterior, 2) AS ventas_mes_anterior,
  ROUND((ventas_netas - ventas_mes_anterior) * 100 / NULLIF(ventas_mes_anterior, 0), 2) AS crecimiento_pct
FROM con_mes_anterior
ORDER BY mes;

### 24. Primera y última compra por cliente

Para cada cliente con pedidos, muestra la fecha de su primera compra y la fecha de su compra más reciente.

In [ ]:
%sql
-- Un pedido cancelado no es una compra, por eso se aplica la regla de venta válida
SELECT
  c.cliente_id,
  c.nombre_cliente,
  MIN(p.fecha_pedido) AS primera_compra,
  MAX(p.fecha_pedido) AS ultima_compra
FROM clientes AS c
JOIN pedidos AS p
  ON c.cliente_id = p.cliente_id
WHERE p.estado <> 'cancelado'
GROUP BY c.cliente_id, c.nombre_cliente
ORDER BY c.cliente_id;

### 25. Clientes con compras válidas en al menos 3 meses distintos

Lista los clientes que realizaron pedidos válidos en al menos 3 meses distintos.

In [ ]:
%sql
SELECT
  c.cliente_id,
  c.nombre_cliente,
  COUNT(DISTINCT date_format(p.fecha_pedido, 'yyyy-MM')) AS meses_con_compra
FROM clientes AS c
JOIN pedidos AS p
  ON c.cliente_id = p.cliente_id
WHERE p.estado <> 'cancelado'
GROUP BY c.cliente_id, c.nombre_cliente
HAVING COUNT(DISTINCT date_format(p.fecha_pedido, 'yyyy-MM')) >= 3
ORDER BY meses_con_compra DESC, c.cliente_id;

### 26. Pedidos con importe superior al promedio

Muestra los pedidos válidos cuyo importe total sea superior al importe promedio de todos los pedidos válidos.

In [ ]:
%sql
WITH totales_pedido AS (
  SELECT
    p.pedido_id,
    p.cliente_id,
    p.fecha_pedido,
    SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)) AS importe_total
  FROM pedidos AS p
  JOIN detalle_pedidos AS d
    ON p.pedido_id = d.pedido_id
  WHERE p.estado <> 'cancelado'
  GROUP BY p.pedido_id, p.cliente_id, p.fecha_pedido
),
promedio AS (
  SELECT AVG(importe_total) AS importe_promedio
  FROM totales_pedido
)
SELECT
  t.pedido_id,
  t.cliente_id,
  t.fecha_pedido,
  ROUND(t.importe_total, 2) AS importe_total,
  ROUND(pm.importe_promedio, 2) AS importe_promedio
FROM totales_pedido AS t
CROSS JOIN promedio AS pm
WHERE t.importe_total > pm.importe_promedio
ORDER BY t.importe_total DESC;

### 27. Productos con precio superior al promedio de su categoría

Lista los productos cuyo precio de lista sea superior al precio promedio de su propia categoría.

In [ ]:
%sql
-- AVG() OVER (PARTITION BY ...) calcula el promedio de la categoría sin colapsar las filas
WITH productos_con_promedio AS (
  SELECT
    producto_id,
    nombre_producto,
    categoria,
    precio_lista,
    AVG(precio_lista) OVER (PARTITION BY categoria) AS precio_promedio_categoria
  FROM productos
)
SELECT
  producto_id,
  nombre_producto,
  categoria,
  precio_lista,
  ROUND(precio_promedio_categoria, 2) AS precio_promedio_categoria
FROM productos_con_promedio
WHERE precio_lista > precio_promedio_categoria
ORDER BY categoria, precio_lista DESC;

### 28. Participación de cada categoría en las ventas totales

Calcula el porcentaje de participación de cada categoría sobre las ventas netas válidas totales.

In [ ]:
%sql
-- SUM() OVER () sin PARTITION BY devuelve el total general en cada fila
WITH ventas_categoria AS (
  SELECT
    pr.categoria,
    SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)) AS ventas_netas
  FROM pedidos AS p
  JOIN detalle_pedidos AS d
    ON p.pedido_id = d.pedido_id
  JOIN productos AS pr
    ON d.producto_id = pr.producto_id
  WHERE p.estado <> 'cancelado'
  GROUP BY pr.categoria
)
SELECT
  categoria,
  ROUND(ventas_netas, 2) AS ventas_netas,
  ROUND(ventas_netas * 100 / SUM(ventas_netas) OVER (), 2) AS participacion_pct
FROM ventas_categoria
ORDER BY participacion_pct DESC;

### 29. Método de pago más usado por ciudad de envío

Identifica el método de pago más utilizado en cada ciudad de envío. Si hay empate, elige alfabéticamente el método de pago.

In [ ]:
%sql
-- ROW_NUMBER desempata por metodo_pago ASC (alfabético); QUALIFY filtra el resultado de la ventana
WITH usos_metodo AS (
  SELECT
    ciudad_envio,
    metodo_pago,
    COUNT(*) AS cantidad_pedidos
  FROM pedidos
  WHERE estado <> 'cancelado'
  GROUP BY ciudad_envio, metodo_pago
)
SELECT
  ciudad_envio,
  metodo_pago,
  cantidad_pedidos
FROM usos_metodo
QUALIFY ROW_NUMBER() OVER (PARTITION BY ciudad_envio ORDER BY cantidad_pedidos DESC, metodo_pago ASC) = 1
ORDER BY ciudad_envio;

### 30. Días promedio entre registro y primera compra por segmento

Calcula, por segmento de cliente, el promedio de días transcurridos entre la fecha de registro y la primera compra.

In [ ]:
%sql
WITH primera_compra AS (
  SELECT
    cliente_id,
    MIN(fecha_pedido) AS fecha_primera_compra
  FROM pedidos
  WHERE estado <> 'cancelado'
  GROUP BY cliente_id
)
SELECT
  c.segmento,
  COUNT(*) AS clientes_con_compra,
  ROUND(AVG(datediff(pc.fecha_primera_compra, c.fecha_registro)), 2) AS dias_promedio_hasta_primera_compra
FROM clientes AS c
JOIN primera_compra AS pc
  ON c.cliente_id = pc.cliente_id
GROUP BY c.segmento
ORDER BY dias_promedio_hasta_primera_compra;